# Data Exploration: Big Cycles 1975–Present

First look at the historical data to identify major regime shifts, cycles, and relationships.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from src.data_fetcher import load_series, load_config
from src import indicators

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 100

# NBER recession dates (approximate) for shading
RECESSIONS = [
    ('1973-11-01', '1975-03-01'),
    ('1980-01-01', '1980-07-01'),
    ('1981-07-01', '1982-11-01'),
    ('1990-07-01', '1991-03-01'),
    ('2001-03-01', '2001-11-01'),
    ('2007-12-01', '2009-06-01'),
    ('2020-02-01', '2020-04-01'),
]

def add_recessions(ax):
    for start, end in RECESSIONS:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.15, color='gray')

print('Loaded.')

## 1. Interest Rates & the Great Disinflation

The defining macro story from 1975–2020: rates peaked ~1981 and fell for 40 years.

In [ ]:
fed_funds = load_series('fred', 'FEDFUNDS').squeeze()
gs10 = load_series('fred', 'DGS10').squeeze()
gs2 = load_series('fred', 'GS2').squeeze()
cpi = load_series('fred', 'CPIAUCSL').squeeze()

# Compute YoY inflation
inflation_yoy = indicators.yoy_change(cpi)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

# Rates
gs10_monthly = gs10.resample('ME').last()
ax1.plot(fed_funds.index, fed_funds, label='Fed Funds', linewidth=0.8)
ax1.plot(gs10_monthly.index, gs10_monthly, label='10Y Treasury', linewidth=0.8)
ax1.plot(gs2.index, gs2, label='2Y Treasury', linewidth=0.8, alpha=0.7)
add_recessions(ax1)
ax1.set_ylabel('Rate (%)')
ax1.set_title('Interest Rates (1975–Present)')
ax1.legend()

# Inflation
ax2.plot(inflation_yoy.index, inflation_yoy, label='CPI YoY %', color='red', linewidth=0.8)
ax2.axhline(y=2, color='gray', linestyle='--', alpha=0.5, label='2% target')
add_recessions(ax2)
ax2.set_ylabel('YoY Change (%)')
ax2.set_title('Inflation (CPI Year-over-Year)')
ax2.legend()

plt.tight_layout()
plt.show()

## 2. Yield Curve & Recession Prediction

Every recession since 1975 was preceded by a yield curve inversion.

In [ ]:
t10y2y = load_series('fred', 'T10Y2Y').squeeze()
t10y2y_monthly = t10y2y.resample('ME').last()

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(t10y2y_monthly.index, t10y2y_monthly, color='navy', linewidth=0.8)
ax.axhline(y=0, color='red', linestyle='--', alpha=0.7)
ax.fill_between(t10y2y_monthly.index, t10y2y_monthly, 0,
                where=t10y2y_monthly < 0, alpha=0.3, color='red', label='Inverted')
add_recessions(ax)
ax.set_ylabel('Spread (%)')
ax.set_title('Yield Curve: 10Y − 2Y Treasury Spread')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Debt Dynamics

The long-term debt cycle: how government and corporate debt have grown relative to GDP.

In [ ]:
fed_debt_gdp = load_series('fred', 'GFDEGDQ188S').squeeze()
corp_debt = load_series('fred', 'BCNSDODNS').squeeze()
gdp = load_series('fred', 'GDPC1').squeeze()

# Corporate debt as % of GDP (both in billions)
corp_debt_gdp = (corp_debt / gdp * 100).dropna()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

ax1.plot(fed_debt_gdp.index, fed_debt_gdp, color='darkred', linewidth=1.2)
add_recessions(ax1)
ax1.set_ylabel('% of GDP')
ax1.set_title('Federal Government Debt / GDP')

ax2.plot(corp_debt_gdp.index, corp_debt_gdp, color='darkblue', linewidth=1.2)
add_recessions(ax2)
ax2.set_ylabel('% of GDP (approx)')
ax2.set_title('Nonfinancial Corporate Debt / GDP')

plt.tight_layout()
plt.show()

# Debt acceleration
debt_accel = indicators.debt_acceleration(fed_debt_gdp)
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(debt_accel.index, debt_accel, width=80, color=np.where(debt_accel > 0, 'red', 'green'), alpha=0.6)
add_recessions(ax)
ax.set_ylabel('YoY Change in Debt/GDP (%)')
ax.set_title('Federal Debt Acceleration (debt growing faster than economy = red)')
ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

## 4. Monetary Policy Regime

M2 growth, monetary base, and real interest rates tell the story of how policymakers responded.

In [ ]:
m2 = load_series('fred', 'M2SL').squeeze()
monetary_base = load_series('fred', 'BOGMBASE').squeeze()

m2_growth = indicators.money_supply_growth(m2)
base_growth = indicators.monetary_base_expansion(monetary_base)
real_rates = indicators.real_rate(fed_funds, inflation_yoy)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(m2_growth.index, m2_growth, color='teal', linewidth=0.8)
axes[0].axhline(0, color='black', linewidth=0.5)
add_recessions(axes[0])
axes[0].set_ylabel('YoY %')
axes[0].set_title('M2 Money Supply Growth')

axes[1].plot(base_growth.index, base_growth, color='purple', linewidth=0.8)
axes[1].axhline(0, color='black', linewidth=0.5)
add_recessions(axes[1])
axes[1].set_ylabel('YoY %')
axes[1].set_title('Monetary Base Growth (QE episodes visible as spikes)')

axes[2].plot(real_rates.index, real_rates, color='darkred', linewidth=0.8)
axes[2].axhline(0, color='gray', linestyle='--', alpha=0.7)
axes[2].fill_between(real_rates.index, real_rates, 0,
                     where=real_rates < 0, alpha=0.3, color='red', label='Negative real rate')
add_recessions(axes[2])
axes[2].set_ylabel('Rate (%)')
axes[2].set_title('Real Fed Funds Rate (nominal − CPI YoY)')
axes[2].legend()

plt.tight_layout()
plt.show()

## 5. Asset Prices: Stocks, Gold, Dollar

How major asset classes have performed across these regime shifts.

In [ ]:
sp500 = load_series('yahoo', '^GSPC')['Close']
gold = load_series('yahoo', 'GC=F')['Close']
usd = load_series('yahoo', 'DX-Y.NYB')['Close']

# Normalize to 100 at each series start for comparison
def normalize(s):
    return s / s.iloc[0] * 100

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# S&P 500 log scale
ax1.semilogy(sp500.index, sp500, color='green', linewidth=0.8, label='S&P 500')
add_recessions(ax1)
ax1.set_ylabel('Price (log scale)')
ax1.set_title('S&P 500 (1975–Present, Log Scale)')
ax1.legend()

# Gold and USD
gold_monthly = gold.resample('ME').last()
usd_monthly = usd.resample('ME').last()
ax2.plot(gold_monthly.index, normalize(gold_monthly), label='Gold (normalized)', color='goldenrod', linewidth=0.8)
ax2_twin = ax2.twinx()
ax2_twin.plot(usd_monthly.index, usd_monthly, label='USD Index', color='steelblue', linewidth=0.8)
add_recessions(ax2)
ax2.set_ylabel('Gold (normalized to 100)')
ax2_twin.set_ylabel('USD Index')
ax2.set_title('Gold vs US Dollar')
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2)

plt.tight_layout()
plt.show()

## 6. Credit Stress Indicators

Corporate bond spreads and loan delinquencies as early warning signals.

In [ ]:
baa_spread = load_series('fred', 'BAA10Y').squeeze()
baa_monthly = baa_spread.resample('ME').last().dropna()

delinquency = load_series('fred', 'DRCCLACBS').squeeze()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

ax1.plot(baa_monthly.index, baa_monthly, color='darkred', linewidth=0.8)
add_recessions(ax1)
ax1.set_ylabel('Spread (%)')
ax1.set_title('BAA Corporate Bond Spread over 10Y Treasury')

ax2.plot(delinquency.index, delinquency, color='orange', linewidth=1.0)
add_recessions(ax2)
ax2.set_ylabel('Rate (%)')
ax2.set_title('Consumer Loan Delinquency Rate')

plt.tight_layout()
plt.show()

## 7. Regime Classification (First Pass)

A simple heuristic to label macro regimes — this is a starting point to iterate on.

In [ ]:
# Build regime inputs
yield_curve = t10y2y.copy()
debt_accel_series = indicators.debt_acceleration(fed_debt_gdp)
real_rate_series = indicators.real_rate(fed_funds, inflation_yoy)

regimes = indicators.regime_classifier(
    yield_curve=yield_curve,
    inflation_yoy=inflation_yoy,
    debt_accel=debt_accel_series,
    real_rate_series=real_rate_series,
)

# Find dominant regime at each point
dominant = regimes.idxmax(axis=1)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Stacked area of regime scores
colors = {'expansion': 'green', 'overheating': 'orange', 'contraction': 'red', 'reflation': 'blue'}
regimes.plot.area(ax=ax1, color=[colors[c] for c in regimes.columns], alpha=0.6, linewidth=0)
add_recessions(ax1)
ax1.set_title('Regime Scores (stacked)')
ax1.set_ylabel('Score')

# Dominant regime over time
regime_colors = dominant.map(colors)
for regime, color in colors.items():
    mask = dominant == regime
    ax2.scatter(dominant.index[mask], [regime] * mask.sum(), c=color, s=2, label=regime)
add_recessions(ax2)
ax2.set_title('Dominant Regime Over Time')
ax2.legend(markerscale=5)

plt.tight_layout()
plt.show()

# Summary stats
print('\nRegime distribution:')
print(dominant.value_counts(normalize=True).round(3) * 100)

## 8. Data Coverage Summary

What data is available at each point in time — important for walk-forward backtesting.

In [ ]:
config = load_config()
coverage = []

for source in ['fred', 'yahoo']:
    for sid, meta in config.get(source, {}).items():
        try:
            df = load_series(source, sid)
            coverage.append({
                'source': source,
                'id': sid,
                'name': meta['name'],
                'category': meta.get('category', ''),
                'start': df.index.min(),
                'end': df.index.max(),
                'rows': len(df),
            })
        except FileNotFoundError:
            coverage.append({
                'source': source,
                'id': sid,
                'name': meta['name'],
                'category': meta.get('category', ''),
                'start': None, 'end': None, 'rows': 0,
            })

cov_df = pd.DataFrame(coverage).sort_values('start')

# Timeline visualization
fig, ax = plt.subplots(figsize=(14, 10))
available = cov_df[cov_df['rows'] > 0].reset_index(drop=True)
for i, row in available.iterrows():
    ax.barh(i, (row['end'] - row['start']).days,
            left=row['start'], height=0.7, alpha=0.7,
            color=plt.cm.Set3(hash(row['category']) % 12))
ax.set_yticks(range(len(available)))
ax.set_yticklabels([f"{r['id']} ({r['name']})" for _, r in available.iterrows()], fontsize=7)
ax.set_xlabel('Date')
ax.set_title('Data Coverage Timeline')
ax.axvline(pd.Timestamp('1975-01-01'), color='red', linestyle='--', alpha=0.5, label='Backtest start')
ax.legend()
plt.tight_layout()
plt.show()